# Sprint 3 — Type Transformers

Spec: [`sprint-3-tasks.md`](../../litemapper/docs/requirements/sprint-3-tasks.md) — 14 tasks (S3-T00..T13) that shipped the built-in `ITypeTransformer` registry.

Conventions (Sprint 2) decide *which* origin member feeds *which* target member. Transformers decide *how* to convert the origin value when the two member types differ. The registry contains 12+ built-ins covering the typical .NET primitive / BCL shapes plus a plug-in point for custom transformers.

| Scope                                            | Task   | Example                                             |
| ------------------------------------------------ | ------ | --------------------------------------------------- |
| `ParsableTransformer`                            | S3-T01 | `string "42"` → `int 42`                            |
| `ToStringTransformer`                            | S3-T02 | `int 42` → `"42"`                                    |
| `DateTime*` family                               | S3-T03 | `DateTime` ↔ `DateTimeOffset` / `DateOnly`          |
| `EnumTransformers`                               | S3-T04 | `enum` ↔ `string`; `enum` ↔ `enum` (by name)        |
| `NullableTransformer`                            | S3-T08 | `int?` ↔ `int`                                       |
| `Base64Transformer`                              | S3-T10 | `byte[]` ↔ `Base64 string`                          |
| Custom `ITypeTransformer<TFrom, TTo>`            | S3-T12 | opt-in via `options.AddTransformer<T>()`            |


## Setup


In [2]:
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;
Console.WriteLine("Ready.");


Ready.


## 1. `ParsableTransformer` + `ToStringTransformer` — string ↔ primitive

The registry picks `ParsableTransformer` when the origin is `string` and the target is a type that implements `IParsable<T>` (every primitive, `Guid`, `DateTime`, `DateTimeOffset`, `decimal`, etc.). The reverse direction uses `ToStringTransformer`.


In [3]:
public sealed class StringInput
{
    public string Id        { get; init; } = "";
    public string Price     { get; init; } = "";
    public string Activated { get; init; } = "";
    public string Uid       { get; init; } = "";
}

public sealed class TypedDto
{
    public int      Id        { get; set; }
    public decimal  Price     { get; set; }
    public DateTime Activated { get; set; }
    public Guid     Uid       { get; set; }
}

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<StringInput, TypedDto>(_ => { }))
    .Forge();

var dto = sculptor.Map<StringInput, TypedDto>(new StringInput
{
    Id        = "42",
    Price     = "19.95",
    Activated = "2026-04-22T09:30:00",
    Uid       = "11111111-2222-3333-4444-555555555555",
});

Console.WriteLine($"Id       : {dto.Id}        ({dto.Id.GetType().Name})");
Console.WriteLine($"Price    : {dto.Price}     ({dto.Price.GetType().Name})");
Console.WriteLine($"Activated: {dto.Activated:u} ({dto.Activated.GetType().Name})");
Console.WriteLine($"Uid      : {dto.Uid}       ({dto.Uid.GetType().Name})");


Error: SmartMapp.Net.Compilation.MappingCompilationException: No conversion available from 'String' to 'Int32'. Register a custom ITypeTransformer<String, Int32> or configure the property explicitly.
   at SmartMapp.Net.Compilation.TransformerExpressionHelper.BuildTransformExpression(Expression originValueExpr, Type originType, Type targetType, ParameterExpression scopeParam, ITypeTransformer transformer) in /_/src/SmartMapp.Net/Compilation/TransformerExpressionHelper.cs:line 93
   at SmartMapp.Net.Compilation.PropertyAssignmentBuilder.BuildValueExpression(PropertyLink link, Expression originParam, ParameterExpression scopeParam, Func`5 nestedMapper) in /_/src/SmartMapp.Net/Compilation/PropertyAssignmentBuilder.cs:line 215
   at SmartMapp.Net.Compilation.PropertyAssignmentBuilder.BuildMemberBindings(IReadOnlyList`1 links, Expression originParam, ParameterExpression scopeParam, HashSet`1 consumedByConstructor, Boolean initOnlyOnly) in /_/src/SmartMapp.Net/Compilation/PropertyAssignme

## 2. `EnumToStringTransformer` / `EnumToEnumTransformer`

Enums round-trip to strings by name (case-insensitive). Two different enums with overlapping member names interconvert by name.


In [4]:
public enum OrderStatus { Pending, Shipped, Delivered, Cancelled }
public enum ExternalOrderState { Pending, Shipped, Delivered, Cancelled }

public sealed class E_DomainOrder { public OrderStatus Status { get; init; } }
public sealed class E_WireOrder   { public string        Status { get; set; } = ""; }
public sealed class E_ExternalOrd { public ExternalOrderState Status { get; set; } }

var s1 = new SculptorBuilder().Configure(o => o.Bind<E_DomainOrder, E_WireOrder>(_ => { })).Forge();
var s2 = new SculptorBuilder().Configure(o => o.Bind<E_DomainOrder, E_ExternalOrd>(_ => { })).Forge();

var src = new E_DomainOrder { Status = OrderStatus.Shipped };
Console.WriteLine($"Domain.Status={src.Status} →");
Console.WriteLine($"  Wire.Status    = \"{s1.Map<E_DomainOrder, E_WireOrder>(src).Status}\"");
Console.WriteLine($"  External.Status= {s2.Map<E_DomainOrder, E_ExternalOrd>(src).Status}  ({s2.Map<E_DomainOrder, E_ExternalOrd>(src).Status.GetType().Name})");


Domain.Status=Shipped →
  Wire.Status    = "Shipped"
  External.Status= Shipped  (ExternalOrderState)


## 3. `DateTimeToDateTimeOffsetTransformer` + `DateOnly` / `TimeOnly`

The `DateTime*` family handles the six common conversions across the BCL date/time types.


In [5]:
public sealed class Event        { public DateTime       At { get; init; } }
public sealed class EventDtoOffset { public DateTimeOffset At { get; set; } }
public sealed class EventDtoDate   { public DateOnly       At { get; set; } }
public sealed class EventDtoTime   { public TimeOnly       At { get; set; } }

var baseline = new DateTime(2026, 4, 22, 14, 30, 0, DateTimeKind.Utc);
var src = new Event { At = baseline };

var sOff  = new SculptorBuilder().Configure(o => o.Bind<Event, EventDtoOffset>(_ => { })).Forge();
var sDate = new SculptorBuilder().Configure(o => o.Bind<Event, EventDtoDate>(_ => { })).Forge();
var sTime = new SculptorBuilder().Configure(o => o.Bind<Event, EventDtoTime>(_ => { })).Forge();

Console.WriteLine($"DateTime       : {baseline:u}");
Console.WriteLine($"DateTimeOffset : {sOff.Map<Event, EventDtoOffset>(src).At:u}");
Console.WriteLine($"DateOnly       : {sDate.Map<Event, EventDtoDate>(src).At}");
Console.WriteLine($"TimeOnly       : {sTime.Map<Event, EventDtoTime>(src).At}");


Error: SmartMapp.Net.Compilation.MappingCompilationException: No conversion available from 'DateTime' to 'DateOnly'. Register a custom ITypeTransformer<DateTime, DateOnly> or configure the property explicitly.
   at SmartMapp.Net.Compilation.TransformerExpressionHelper.BuildTransformExpression(Expression originValueExpr, Type originType, Type targetType, ParameterExpression scopeParam, ITypeTransformer transformer) in /_/src/SmartMapp.Net/Compilation/TransformerExpressionHelper.cs:line 93
   at SmartMapp.Net.Compilation.PropertyAssignmentBuilder.BuildValueExpression(PropertyLink link, Expression originParam, ParameterExpression scopeParam, Func`5 nestedMapper) in /_/src/SmartMapp.Net/Compilation/PropertyAssignmentBuilder.cs:line 215
   at SmartMapp.Net.Compilation.PropertyAssignmentBuilder.BuildSingleAssignment(PropertyLink link, Expression originParam, Expression targetVar, ParameterExpression scopeParam, Func`5 nestedMapper) in /_/src/SmartMapp.Net/Compilation/PropertyAssignmentBu

## 4. `Base64Transformer` — `byte[]` ↔ Base64 `string`

Both directions register: serialise binary to wire-safe strings and parse them back.


In [6]:
public sealed class Avatar    { public byte[] Thumbnail { get; init; } = Array.Empty<byte>(); }
public sealed class AvatarDto { public string Thumbnail { get; set; } = ""; }
public sealed class AvatarRev { public byte[] Thumbnail { get; set; } = Array.Empty<byte>(); }

var src = new Avatar { Thumbnail = new byte[] { 0x48, 0x65, 0x6C, 0x6C, 0x6F } };

var encodeSculptor = new SculptorBuilder().Configure(o => o.Bind<Avatar, AvatarDto>(_ => { })).Forge();
var decodeSculptor = new SculptorBuilder().Configure(o => o.Bind<AvatarDto, AvatarRev>(_ => { })).Forge();

var encoded = encodeSculptor.Map<Avatar, AvatarDto>(src);
var decoded = decodeSculptor.Map<AvatarDto, AvatarRev>(encoded);

Console.WriteLine($"Original bytes : [{string.Join(", ", src.Thumbnail)}]");
Console.WriteLine($"Base64         : \"{encoded.Thumbnail}\"");
Console.WriteLine($"Round-tripped  : [{string.Join(", ", decoded.Thumbnail)}]");
Console.WriteLine($"Lossless?      : {src.Thumbnail.SequenceEqual(decoded.Thumbnail)}");


Error: SmartMapp.Net.Compilation.MappingCompilationException: No conversion available from 'Byte[]' to 'String'. Register a custom ITypeTransformer<Byte[], String> or configure the property explicitly.
   at SmartMapp.Net.Compilation.TransformerExpressionHelper.BuildTransformExpression(Expression originValueExpr, Type originType, Type targetType, ParameterExpression scopeParam, ITypeTransformer transformer) in /_/src/SmartMapp.Net/Compilation/TransformerExpressionHelper.cs:line 93
   at SmartMapp.Net.Compilation.PropertyAssignmentBuilder.BuildValueExpression(PropertyLink link, Expression originParam, ParameterExpression scopeParam, Func`5 nestedMapper) in /_/src/SmartMapp.Net/Compilation/PropertyAssignmentBuilder.cs:line 215
   at SmartMapp.Net.Compilation.PropertyAssignmentBuilder.BuildMemberBindings(IReadOnlyList`1 links, Expression originParam, ParameterExpression scopeParam, HashSet`1 consumedByConstructor, Boolean initOnlyOnly) in /_/src/SmartMapp.Net/Compilation/PropertyAssign

## 5. Custom `ITypeTransformer<TFrom, TTo>` registration

When a domain-specific conversion isn't in the built-in set, implement the interface and register with `options.AddTransformer<T>()`. The registry resolves the most-specific match per link.


In [7]:
// Domain-specific Money value object.
public sealed record Money(decimal Amount, string Currency)
{
    public override string ToString() => $"{Amount:0.00} {Currency}";
}

public sealed class MoneyToStringTransformer : ITypeTransformer<Money, string>
{
    public string Transform(Money origin, MappingScope scope)
        => origin is null ? "" : $"{origin.Amount:0.00} {origin.Currency}";

    public bool CanTransform(Type originType, Type targetType)
        => originType == typeof(Money) && targetType == typeof(string);
}

public sealed class Invoice    { public Money  Total { get; init; } = new(0m, "USD"); }
public sealed class InvoiceDto { public string Total { get; set; } = ""; }

var sculptor = new SculptorBuilder()
    .AddTransformer<MoneyToStringTransformer>()
    .Configure(o => o.Bind<Invoice, InvoiceDto>(_ => { }))
    .Forge();

var inv = new Invoice { Total = new Money(149.95m, "USD") };
var dto = sculptor.Map<Invoice, InvoiceDto>(inv);
Console.WriteLine($"Invoice.Total : {inv.Total}");
Console.WriteLine($"Dto.Total     : \"{dto.Total}\"");


Error: SmartMapp.Net.Compilation.MappingCompilationException: No conversion available from 'Money' to 'String'. Register a custom ITypeTransformer<Money, String> or configure the property explicitly.
   at SmartMapp.Net.Compilation.TransformerExpressionHelper.BuildTransformExpression(Expression originValueExpr, Type originType, Type targetType, ParameterExpression scopeParam, ITypeTransformer transformer) in /_/src/SmartMapp.Net/Compilation/TransformerExpressionHelper.cs:line 93
   at SmartMapp.Net.Compilation.PropertyAssignmentBuilder.BuildValueExpression(PropertyLink link, Expression originParam, ParameterExpression scopeParam, Func`5 nestedMapper) in /_/src/SmartMapp.Net/Compilation/PropertyAssignmentBuilder.cs:line 215
   at SmartMapp.Net.Compilation.PropertyAssignmentBuilder.BuildMemberBindings(IReadOnlyList`1 links, Expression originParam, ParameterExpression scopeParam, HashSet`1 consumedByConstructor, Boolean initOnlyOnly) in /_/src/SmartMapp.Net/Compilation/PropertyAssignme

## Next

- **`sprint-04-construction-and-graph.ipynb`** — once values are resolved, how is the target instance constructed? Records, `init`-only, circular-reference tracking, depth limits.
